In [13]:
import torch 
import matplotlib.pyplot as plt
import random
import numpy as np
from matplotlib.colors import ListedColormap
import dill
from sparse_generalization.envs.box_world.env import BoxWorldEnv
from sparse_generalization.envs.box_world.wrappers import make_env
from minigrid.wrappers import FullyObsWrapper
import gymnasium as gym
import cv2
import matplotlib.pyplot as plt
from torchmetrics.classification.accuracy import BinaryAccuracy
gym.register('BoxWorldEnv-v1', BoxWorldEnv)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
with open('../results/bern_mha_20seeds_24_Feb_2026__06h27m.pl', 'rb') as file:
    results = dill.load(file)

In [47]:
A = torch.eye(100) 
num_rand = 1

for _ in range(4):
    I = torch.eye(100) * 4
    indices = torch.randint(low=0, high=100, size=(2*num_rand,)).reshape(2, num_rand)
    print(indices)
    A[indices[0], indices[1]] = 1
    A = A @ I
    
A.sum()

tensor([[26],
        [43]])
tensor([[19],
        [ 5]])
tensor([[59],
        [60]])
tensor([[78],
        [22]])


tensor(25940.)

In [25]:
with open('../data/box_world/balanced_dist_500.pl', 'rb') as file:
    dataset = dill.load(file)
    file.close()

In [26]:
x_train = dataset['X_train']
y_train = dataset['Y_train']
x_test_ind = dataset['X_test_ind']
y_test_ind = dataset['Y_test_ind']
x_test_ood = dataset['X_test_ood']
y_test_ood = dataset['Y_test_ood']
edges_train = dataset['edges_train']
edges_test = dataset['edges_test_ood']

In [195]:
test = edges_train[0:50]
test = test.view(50 * 1 * 4, 2, 2)
test = test[:, :, 1] * 10 + test[:, :, 0] 
test = test.view(50, 4, 2)
A = torch.zeros(50, 100, 100)
item = test[0]
A[0, item[:, 0], item[:, 1]] = 1

In [190]:
test[0]

tensor([[[16, 54],
         [11, 12],
         [51, 52],
         [27, 26]]])

In [203]:
edges_train[3]

tensor([[[[6, 7],
          [5, 8]],

         [[4, 6],
          [5, 6]],

         [[1, 7],
          [2, 7]],

         [[8, 5],
          [7, 5]]]])

In [145]:
def get_edges(obs, residual=False):
    edges = []
    # get goal-lock 
    x_goal, y_goal = torch.where(obs[:, :, 0] == 8)
    goal = torch.stack([x_goal, y_goal], dim=1).squeeze()
    edges.append(((goal[0]+1, goal[1]), (goal[0], goal[1])))
    # for each key check key + 1 in lock then a combo
    xs, ys = torch.where(x_train[0][:, :, 0] == 13)
    locks = torch.stack([xs, ys], dim=1)
    locks = locks[~(locks == torch.tensor([goal[0]+1, goal[1]])).all(dim=1)]
    
    xs, ys = torch.where(x_train[0][:, :, 0] == 12)
    keys = torch.stack([xs, ys], dim=1)
    for lock in locks:
        key = (lock[0]-1, lock[1])
        edges.append(((lock[0], lock[1]), (key)))
        keys = keys[~(keys == torch.tensor([lock[0]-1, lock[1]])).all(dim=1)]
        
    # remaining key is residual or looking from agent
    first_key = keys.squeeze()
    if not residual: 
        xs, ys = torch.where(x_train[0][:, :, 0] == 10)
        agent = torch.stack([xs, ys], dim=1).squeeze()
        edges.append(((agent[0], agent[1]), (first_key[0], first_key[1])))
    
    return edges

In [43]:
from sparse_generalization.layers.oracle import MultiHeadAttentionOracle

mha = MultiHeadAttentionOracle(embed_size=3, num_heads=3)
x_train = x_train.view(500, 100, 3).float()
mha(x_train, x_train, x_train, edges_train)[1].shape

torch.Size([500, 100, 100])